# Processamento de Língua Natural - EP

## SetFit com mineração de hard negatives

O modelo classifica as respostas nas classes `c1`, `c234` e `c5` usando SetFit.

O ajuste dos embeddings é feito com `BatchHardTripletLoss`. Para cada âncora, a loss considera exemplos positivos e negativos difíceis dentro do batch, aproximando textos da mesma classe e afastando textos de classes diferentes.

Após o ajuste contrastivo, um classificador é treinado sobre os embeddings produzidos pelo modelo.

## Bibliotecas

No Google Colab, as dependências podem ser instaladas com:

```python
%pip install -q setfit==1.1.3 datasets sentence-transformers pandas openpyxl scikit-learn
```

In [ ]:
import gc
import random
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold

from sentence_transformers.losses import BatchHardTripletLoss

from setfit import (
    SetFitModel,
    Trainer,
    TrainingArguments,
)

## Configuração

O backbone utilizado é o `paraphrase-multilingual-mpnet-base-v2`, um Sentence Transformer multilíngue com suporte a português.

A validação usa 3 folds estratificados. O ajuste contrastivo é executado por 1 época, com textos limitados a 128 tokens.

In [ ]:
SEED = 42

MODELO_BASE = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

N_SPLITS = 3
MAX_LENGTH = 128

# Treinamento contrastivo do corpo do SetFit.
EMBEDDING_EPOCHS = 1
BATCH_SIZE = 18
BODY_LEARNING_RATE = 2e-5

# BatchHardTripletLoss:
# cada âncora usa o positivo mais difícil e o negativo
# mais difícil disponíveis dentro do batch.
TRIPLET_MARGIN = 0.30
SAMPLES_PER_LABEL = 2

# Classificador final do SetFit.
HEAD_MAX_ITER = 1000
HEAD_CLASS_WEIGHT = "balanced"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()

LOG_PATH = "resultados_setfit_hard_negatives.txt"


def fixar_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def log_resultado(mensagem=""):
    mensagem = str(mensagem)
    print(mensagem)

    with open(LOG_PATH, "a", encoding="utf-8") as arquivo:
        arquivo.write(mensagem + "\n")


fixar_seed(SEED)

with open(LOG_PATH, "w", encoding="utf-8") as arquivo:
    arquivo.write("EP1 - SetFit com hard negatives\n")
    arquivo.write("=" * 64 + "\n")
    arquivo.write(
        f"Início: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    )
    arquivo.write(f"Modelo base: {MODELO_BASE}\n")
    arquivo.write(f"Dispositivo: {DEVICE}\n")
    arquivo.write(
        f"Configuração: {N_SPLITS} folds, "
        f"{EMBEDDING_EPOCHS} época(s) contrastiva(s), "
        f"MAX_LENGTH={MAX_LENGTH}, BATCH_SIZE={BATCH_SIZE}\n"
    )
    arquivo.write(
        f"Loss: BatchHardTripletLoss, margin={TRIPLET_MARGIN}, "
        f"samples_per_label={SAMPLES_PER_LABEL}\n"
    )
    arquivo.write(
        f"Head: LogisticRegression, "
        f"class_weight={HEAD_CLASS_WEIGHT}\n"
    )
    arquivo.write("=" * 64 + "\n\n")

log_resultado(f"Dispositivo: {DEVICE}")
log_resultado(f"Modelo: {MODELO_BASE}")
log_resultado(f"Log: {LOG_PATH}")

## Importação dos dados

Os conjuntos de treino e teste são carregados a partir das planilhas fornecidas.

O texto é mantido próximo da forma original, sem remoção de stopwords, stemming ou lematização.

In [ ]:
dados_treino = pd.read_excel(
    "dados/train.xlsx",
    converters={"resp_text": str, "clarity": str},
)

dados_teste = pd.read_excel(
    "dados/test1.xlsx",
    converters={"resp_text": str, "clarity": str},
)

X_treino = (
    dados_treino["resp_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

Y_treino = (
    dados_treino["clarity"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_teste = (
    dados_teste["resp_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

log_resultado()
log_resultado(f"Instâncias de treino: {len(X_treino)}")
log_resultado(f"Instâncias de teste: {len(X_teste)}")
log_resultado("Distribuição das classes no treino:")
log_resultado(
    dados_treino["clarity"]
    .value_counts()
    .to_string()
)

## Pré-processamento e rótulos

O pré-processamento normaliza apenas os espaços em branco.

Os rótulos `c1`, `c234` e `c5` são convertidos para valores inteiros durante o treinamento.

In [ ]:
def preprocess_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


LABEL_TO_INDEX = {
    "c1": 0,
    "c234": 1,
    "c5": 2,
}

INDEX_TO_LABEL = {
    0: "c1",
    1: "c234",
    2: "c5",
}

labels_invalidos = sorted(
    set(Y_treino) - set(LABEL_TO_INDEX)
)

if labels_invalidos:
    raise ValueError(
        f"Rótulos inesperados no treino: {labels_invalidos}"
    )

X_treino_processado = [
    preprocess_text(texto)
    for texto in X_treino
]

X_teste_processado = [
    preprocess_text(texto)
    for texto in X_teste
]

y_indices = np.array(
    [LABEL_TO_INDEX[label] for label in Y_treino],
    dtype=np.int64,
)

## Hard negative mining

A `BatchHardTripletLoss` utiliza triplas formadas por:

- **âncora**: exemplo de referência;
- **positivo**: exemplo da mesma classe;
- **negativo**: exemplo de outra classe.

Dentro de cada batch, são selecionados o positivo mais distante e o negativo mais próximo da âncora. O sampler organiza os batches de forma que haja exemplos suficientes de cada classe para formar as triplas.

## Funções auxiliares

As funções abaixo criam os datasets, inicializam o modelo e executam as predições.

O classificador final é uma regressão logística treinada sobre os embeddings ajustados pelo SetFit.

In [ ]:
def criar_dataset(textos, labels):
    return Dataset.from_dict(
        {
            "text": list(textos),
            "label": [
                int(label)
                for label in labels
            ],
        }
    )


def criar_modelo():
    model = SetFitModel.from_pretrained(
        MODELO_BASE,
        head_params={
            "max_iter": HEAD_MAX_ITER,
            "class_weight": HEAD_CLASS_WEIGHT,
            "random_state": SEED,
        },
    )

    model.to(DEVICE)

    return model


def criar_argumentos(seed):
    return TrainingArguments(
        batch_size=BATCH_SIZE,
        num_epochs=EMBEDDING_EPOCHS,
        body_learning_rate=BODY_LEARNING_RATE,
        loss=BatchHardTripletLoss,
        margin=TRIPLET_MARGIN,
        samples_per_label=SAMPLES_PER_LABEL,
        max_length=MAX_LENGTH,
        use_amp=USE_AMP,
        show_progress_bar=True,
        seed=seed,
        report_to="none",
    )


def prever_indices(model, textos):
    predicoes = model.predict(
        list(textos)
    )

    if torch.is_tensor(predicoes):
        predicoes = (
            predicoes
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        predicoes,
        dtype=np.int64,
    )

## Validação cruzada

A avaliação usa `StratifiedKFold` com 3 folds e `random_state` fixo.

As previsões de validação de cada fold são armazenadas como previsões *out-of-fold* (OOF).

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

resultados = []

predicoes_oof = np.full(
    len(X_treino_processado),
    -1,
    dtype=np.int64,
)

inicio_cv = time.time()

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        X_treino_processado,
        y_indices,
    ),
    start=1,
):
    fixar_seed(SEED + fold)

    inicio_fold = time.time()

    log_resultado()
    log_resultado("=" * 64)
    log_resultado(f"FOLD {fold}/{N_SPLITS}")
    log_resultado("=" * 64)

    textos_fold_treino = [
        X_treino_processado[i]
        for i in train_idx
    ]

    textos_fold_validacao = [
        X_treino_processado[i]
        for i in val_idx
    ]

    y_fold_treino = y_indices[train_idx]
    y_fold_validacao = y_indices[val_idx]

    dataset_fold_treino = criar_dataset(
        textos_fold_treino,
        y_fold_treino,
    )

    model = criar_modelo()

    args = criar_argumentos(
        seed=SEED + fold
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_fold_treino,
        metric="accuracy",
    )

    trainer.train()

    pred_treino = prever_indices(
        model,
        textos_fold_treino,
    )

    pred_validacao = prever_indices(
        model,
        textos_fold_validacao,
    )

    predicoes_oof[val_idx] = (
        pred_validacao
    )

    acc_treino = accuracy_score(
        y_fold_treino,
        pred_treino,
    )

    acc_validacao = accuracy_score(
        y_fold_validacao,
        pred_validacao,
    )

    tempo_fold = (
        time.time() - inicio_fold
    )

    resultados.append(
        {
            "fold": fold,
            "train_accuracy": acc_treino,
            "validation_accuracy": acc_validacao,
            "tempo_segundos": tempo_fold,
        }
    )

    log_resultado(
        f"Acurácia de treino: "
        f"{acc_treino:.2%}"
    )

    log_resultado(
        f"Acurácia de validação: "
        f"{acc_validacao:.2%}"
    )

    log_resultado(
        f"Gap do fold: "
        f"{acc_treino - acc_validacao:.2%}"
    )

    log_resultado(
        f"Tempo do fold: "
        f"{tempo_fold / 60:.2f} min"
    )

    del trainer
    del model
    del dataset_fold_treino

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

resultados_df = pd.DataFrame(
    resultados
)

acuracia_validacao = (
    resultados_df[
        "validation_accuracy"
    ].mean()
)

desvio_validacao = (
    resultados_df[
        "validation_accuracy"
    ].std()
)

acuracia_treino_cv = (
    resultados_df[
        "train_accuracy"
    ].mean()
)

gap = (
    acuracia_treino_cv
    - acuracia_validacao
)

tempo_cv = (
    time.time() - inicio_cv
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "RESUMO DA VALIDAÇÃO CRUZADA"
)
log_resultado("=" * 64)

log_resultado(
    resultados_df.to_string(
        index=False
    )
)

log_resultado()

log_resultado(
    f"Acurácia Média da Validação Cruzada: "
    f"{acuracia_validacao:.2%}"
)

log_resultado(
    f"Desvio-padrão da Validação: "
    f"{desvio_validacao:.2%}"
)

log_resultado(
    f"Acurácia Média de Treino nos Folds: "
    f"{acuracia_treino_cv:.2%}"
)

log_resultado(
    f"Gap Treino-Validação: "
    f"{gap:.2%}"
)

log_resultado(
    f"Tempo total da validação cruzada: "
    f"{tempo_cv / 60:.2f} min"
)

## Matriz de confusão e relatório OOF

A matriz de confusão e o `classification_report` são calculados a partir das previsões OOF.

In [ ]:
ordem_classes = [0, 1, 2]
nomes_classes = [
    "c1",
    "c234",
    "c5",
]

matriz = confusion_matrix(
    y_indices,
    predicoes_oof,
    labels=ordem_classes,
)

relatorio = classification_report(
    y_indices,
    predicoes_oof,
    labels=ordem_classes,
    target_names=nomes_classes,
    digits=4,
    zero_division=0,
)

matriz_df = pd.DataFrame(
    matriz,
    index=[
        f"real_{classe}"
        for classe in nomes_classes
    ],
    columns=[
        f"pred_{classe}"
        for classe in nomes_classes
    ],
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "MATRIZ DE CONFUSÃO OOF"
)
log_resultado("=" * 64)
log_resultado(
    matriz_df.to_string()
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "CLASSIFICATION REPORT OOF"
)
log_resultado("=" * 64)
log_resultado(relatorio)

matriz_df

## Confusões entre classes

Os erros são agrupados pelos pares:

- `c1` e `c234`;
- `c234` e `c5`;
- `c1` e `c5`.

Essa contagem permite observar em quais fronteiras ocorre a maior parte das confusões.

In [ ]:
def contar_confusao(real, previsto, classe_a, classe_b):
    a = LABEL_TO_INDEX[classe_a]
    b = LABEL_TO_INDEX[classe_b]

    return int(
        (
            ((real == a) & (previsto == b))
            |
            ((real == b) & (previsto == a))
        ).sum()
    )


confusoes_fronteira = {
    "c1 <-> c234": contar_confusao(
        y_indices,
        predicoes_oof,
        "c1",
        "c234",
    ),
    "c234 <-> c5": contar_confusao(
        y_indices,
        predicoes_oof,
        "c234",
        "c5",
    ),
    "c1 <-> c5": contar_confusao(
        y_indices,
        predicoes_oof,
        "c1",
        "c5",
    ),
}

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "CONFUSÕES ENTRE PARES DE CLASSES"
)
log_resultado("=" * 64)

for par, quantidade in confusoes_fronteira.items():
    log_resultado(
        f"{par}: {quantidade}"
    )

confusoes_fronteira

## Treinamento final

O modelo final é treinado com todas as instâncias rotuladas do conjunto de treino.

O conjunto de teste é usado somente para predição.

In [ ]:
fixar_seed(SEED)

inicio_treino_final = time.time()

dataset_treino_final = criar_dataset(
    X_treino_processado,
    y_indices,
)

model_final = criar_modelo()

args_final = criar_argumentos(
    seed=SEED
)

trainer_final = Trainer(
    model=model_final,
    args=args_final,
    train_dataset=dataset_treino_final,
    metric="accuracy",
)

trainer_final.train()

pred_treino_final = prever_indices(
    model_final,
    X_treino_processado,
)

acc_treino_final = accuracy_score(
    y_indices,
    pred_treino_final,
)

tempo_treino_final = (
    time.time() - inicio_treino_final
)

log_resultado()
log_resultado("=" * 64)
log_resultado("TREINO FINAL")
log_resultado("=" * 64)

log_resultado(
    f"Acurácia do Treino (modelo final): "
    f"{acc_treino_final:.2%}"
)

log_resultado(
    f"Tempo do treino final: "
    f"{tempo_treino_final / 60:.2f} min"
)

## Predição do conjunto de teste

As classes previstas são convertidas novamente para `c1`, `c234` e `c5`, preservando a ordem das linhas de `test1.xlsx`.

In [ ]:
pred_teste_indices = prever_indices(
    model_final,
    X_teste_processado,
)

Y_previsto = [
    INDEX_TO_LABEL[int(indice)]
    for indice in pred_teste_indices
]

assert len(Y_previsto) == len(X_teste)

assert set(Y_previsto).issubset(
    {"c1", "c234", "c5"}
)

distribuicao_teste = (
    pd.Series(Y_previsto)
    .value_counts()
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "DISTRIBUIÇÃO DAS PREVISÕES DO TESTE"
)
log_resultado("=" * 64)
log_resultado(
    distribuicao_teste.to_string()
)

distribuicao_teste

In [ ]:
teste_rotulado = pd.DataFrame(
    {
        "resp_text": X_teste,
        "clarity": Y_previsto,
    }
)

teste_rotulado.to_excel(
    "teste_rotulado_setfit.xlsx",
    index=False,
)

oof_df = pd.DataFrame(
    {
        "resp_text": X_treino,
        "clarity_real": Y_treino,
        "clarity_pred_oof": [
            INDEX_TO_LABEL[int(indice)]
            for indice in predicoes_oof
        ],
    }
)

oof_df["acertou"] = (
    oof_df["clarity_real"]
    == oof_df["clarity_pred_oof"]
)

oof_df.to_csv(
    "predicoes_oof_setfit.csv",
    index=False,
    encoding="utf-8",
)

log_resultado()
log_resultado(
    "Arquivo de teste salvo: "
    "teste_rotulado_setfit.xlsx"
)

log_resultado(
    "Predições OOF salvas: "
    "predicoes_oof_setfit.csv"
)

log_resultado(
    f"Arquivo de log salvo: {LOG_PATH}"
)

log_resultado(
    f"Fim: "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

teste_rotulado.head()

## Registro dos resultados

A célula abaixo exibe o conteúdo salvo em `resultados_setfit_hard_negatives.txt`.

In [ ]:
with open(
    LOG_PATH,
    "r",
    encoding="utf-8",
) as arquivo:
    print(arquivo.read())